In [ ]:
import json
import torch
import gc
from tqdm import tqdm
from unsloth import FastLanguageModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ==========================================
# 1. FILE PATHS & CONFIGURATION
# ==========================================
input_file = "/workspaces/Arch_PC_Assistent/dataset/benchmark_questions.json"
output_file = "/workspaces/Arch_PC_Assistent/outputs/benchmark/benchmark_results.json"

# WICHTIG: Trage hier den genauen Namen deines ungetunten Basis-Modells ein!
base_model_path = "unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit"
sft_model_path = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1"
rsft_model_path = "/workspaces/Arch_PC_Assistent/outputs/RSFT/arch_assistant_final_lora"

max_seq_length = 4096

# ==========================================
# 2. PREPARE QUESTIONS & RAG CONTEXT
# ==========================================
with open(input_file, "r", encoding="utf-8") as f:
    questions = json.load(f)

print(f"Loaded {len(questions)} questions. Fetching RAG context for all...")

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(embedding_function=embeddings,
                     persist_directory=persist_directory)

benchmark_data = []

for q in tqdm(questions, desc="Pre-fetching RAG context"):
    docs = vectorstore.similarity_search(q, k=3)
    rag_context = "\n\n".join([f"--- Chunk {i+1} ---\n{d.page_content}" for i, d in enumerate(docs)])
    
    benchmark_data.append({
        "question": q,
        "context": rag_context,
        "answers": {
            "base": "",
            "sft": "",
            "rsft": ""
        }
    })

# ==========================================
# 3. SYSTEM PROMPT DEFINITION
# ==========================================
def build_messages(question, context):
    system_prompt = f"""You are ArchAgent, an expert Arch Linux with hyprland and zsh setup assistant. 
Below is retrieved context from the Arch Wiki. Use this context as your primary source of truth to ensure accuracy.

CONTEXT:
{context}

CRITICAL INSTRUCTIONS:
If the context is incomplete or does not fully cover the user's problem, you MUST seamlessly use your own internal knowledge to provide a complete solution.

Your output MUST strictly follow this exact format:
<think>
[Analyze the user's problem. Check what information is available in the context. If something is missing, retrieve it from your internal knowledge. Plan your solution.]
</think>
<answer>
[Your final, clear, and actionable answer here.]
</answer>"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

# ==========================================
# 4. INFERENCE LOOP GENERATOR
# ==========================================
def run_model_inference(model_path, model_key):
    print(f"\n[{model_key.upper()}] Loading model from {model_path}...")
    
    # Modell in den VRAM laden
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = max_seq_length,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)
    
    print(f"[{model_key.upper()}] Starting generation for {len(benchmark_data)} questions...")
    
    for item in tqdm(benchmark_data, desc=f"Generating {model_key}"):
        messages = build_messages(item["question"], item["context"])
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        # Generation (Deterministic settings for fair comparison)
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            use_cache=True,
            temperature=0.3, 
            top_p=0.9
        )
        
        decoded_output = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        item["answers"][model_key] = decoded_output
    
    # ------------------------------------------
    # CRITICAL: VRAM CLEARING
    # ------------------------------------------
    print(f"[{model_key.upper()}] Unloading model and clearing VRAM...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. EXECUTE PIPELINE
# ==========================================
# Wir rufen die Funktion nacheinander für alle drei Modelle auf
run_model_inference(base_model_path, "base")
run_model_inference(sft_model_path, "sft")
run_model_inference(rsft_model_path, "rsft")

# ==========================================
# 6. SAVE RESULTS
# ==========================================
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(benchmark_data, f, indent=4, ensure_ascii=False)

print(f"\nBenchmark completed successfully! Results saved to {output_file}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loaded 50 questions. Fetching RAG context for all...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Pre-fetching RAG context: 100%|██████████| 50/50 [00:00<00:00, 110.85it/s]



[BASE] Loading model from unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[RSFT] Starting generation for 50 questions...


Generating rsft: 100%|██████████| 50/50 [09:54<00:00, 11.90s/it]


[RSFT] Unloading model and clearing VRAM...


FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/Arch_PC_Assistent/output/benchmark/benchmark_results.json'